# Export Topology to GLB Format

This notebook demonstrates how to export topologic_fast geometries to the GLB (binary glTF) format for use in 3D viewers, game engines, AR/VR applications, and web visualizations.

**Note:** GLB export is not yet implemented in topologic_fast. This notebook shows the intended workflow and provides a Python-based implementation that can be used until native support is added.

## What is GLB?

GLB is the binary version of the GL Transmission Format (glTF), a royalty-free specification for efficient transmission and loading of 3D models. It's widely supported by:
- Three.js / Babylon.js (web 3D)
- Unity / Unreal Engine
- Blender
- AR/VR platforms
- Windows 3D Viewer

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
import numpy as np
import json
import os

# Check if pygltflib is available for GLB export
try:
    from pygltflib import (
        GLTF2, Scene, Node, Mesh, Buffer, BufferView, Accessor, Asset, Primitive, Material,
        ARRAY_BUFFER, ELEMENT_ARRAY_BUFFER, FLOAT, UNSIGNED_SHORT, UNSIGNED_INT
    )
    PYGLTFLIB_AVAILABLE = True
    print("pygltflib is available - GLB export enabled")
except ImportError:
    PYGLTFLIB_AVAILABLE = False
    print("pygltflib not installed. Install with: pip install pygltflib")
    print("GLB export will not be available, but visualization will still work.")

## 1. Create Sample Geometry

Let's create various topological objects to export.

In [ ]:
# Create a box
box = tf.Cell.Box(0, 0, 0, 2, 2, 2)
print(f"Box: {len(box.Faces())} faces, {len(box.Edges())} edges, {len(box.Vertices())} vertices")

# Create a cylinder
cylinder = tf.Cell.Cylinder(5, 0, 0, 1.0, 3.0, 32)
print(f"Cylinder: {len(cylinder.Faces())} faces")

# Create a torus-like shape using multiple cylinders
# NOTE: tf.Cell.Torus is not yet implemented
# torus = tf.Cell.Torus(...)  # Not yet implemented in topologic_fast

# Create a sphere
sphere = tf.Cell.Sphere(10, 0, 0, 1.5, 16, 8)
print(f"Sphere: {len(sphere.Faces())} faces")

## 2. Generate Mesh Data

topologic_fast provides mesh generation capabilities that we can use for GLB export.

In [ ]:
# Generate mesh from box
box_mesh = tf.Mesh.ByCell(box)
print(f"Box mesh: {box_mesh.NumVertices()} vertices, {box_mesh.NumTriangles()} triangles")

# Generate mesh from cylinder
cylinder_mesh = tf.Mesh.ByCell(cylinder)
print(f"Cylinder mesh: {cylinder_mesh.NumVertices()} vertices, {cylinder_mesh.NumTriangles()} triangles")

# Generate mesh from sphere
sphere_mesh = tf.Mesh.ByCell(sphere)
print(f"Sphere mesh: {sphere_mesh.NumVertices()} vertices, {sphere_mesh.NumTriangles()} triangles")

## 3. Visualize the Geometry

Before exporting, let's visualize what we're going to export using Plotly.

In [ ]:
def visualize_cell(cell, name="Cell", color='lightblue'):
    """Visualize a cell using plotly."""
    traces = []
    
    faces = cell.Faces()
    for face in faces:
        vertices = face.Vertices()
        coords = [v.Coordinates() for v in vertices]
        
        if len(coords) >= 3:
            x = [c[0] for c in coords]
            y = [c[1] for c in coords]
            z = [c[2] for c in coords]
            
            traces.append(go.Mesh3d(
                x=x, y=y, z=z,
                color=color,
                opacity=0.7,
                alphahull=0,
                name=name,
                showlegend=False
            ))
    
    # Add edges
    edges = cell.Edges()
    for edge in edges:
        edge_verts = edge.Vertices()
        if len(edge_verts) == 2:
            p1 = edge_verts[0].Coordinates()
            p2 = edge_verts[1].Coordinates()
            traces.append(go.Scatter3d(
                x=[p1[0], p2[0]], y=[p1[1], p2[1]], z=[p1[2], p2[2]],
                mode='lines',
                line=dict(color='black', width=2),
                showlegend=False,
                hoverinfo='skip'
            ))
    
    return traces

# Create combined visualization
fig = go.Figure()

# Add each cell with different colors
for trace in visualize_cell(box, "Box", 'lightblue'):
    fig.add_trace(trace)
for trace in visualize_cell(cylinder, "Cylinder", 'lightgreen'):
    fig.add_trace(trace)
for trace in visualize_cell(sphere, "Sphere", 'lightyellow'):
    fig.add_trace(trace)

fig.update_layout(
    title='Geometry to Export to GLB',
    scene=dict(
        aspectmode='data',
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z'
    ),
    width=900,
    height=600
)

fig.show()

## 4. GLB Export Function

Here's a Python implementation that exports topologic_fast geometry to GLB format. This bridges the gap until native GLB support is added to topologic_fast.

**Note:** This implementation uses the `pygltflib` package. Install it with:
```bash
pip install pygltflib
```

In [ ]:
# NOTE: Native GLB export is not yet implemented in topologic_fast
# The following function provides a Python-based implementation

def export_to_glb(
    cells,
    filepath,
    colors=None,
    default_color=(0.5, 0.5, 0.8, 1.0),
    include_edges=False,
    silent=False
):
    """
    Export topologic_fast cells to GLB format.
    
    Parameters:
    -----------
    cells : list or Cell
        Cell or list of Cells to export
    filepath : str
        Output path for .glb file
    colors : list, optional
        List of RGBA colors (0-1 range) for each cell
    default_color : tuple
        Default RGBA color if colors not specified
    include_edges : bool
        Whether to include edge lines in the export
    silent : bool
        Suppress output messages
    """
    if not PYGLTFLIB_AVAILABLE:
        raise ImportError("pygltflib is required for GLB export. Install with: pip install pygltflib")
    
    # Ensure cells is a list
    if not isinstance(cells, (list, tuple)):
        cells = [cells]
    
    # Set default colors
    if colors is None:
        colors = [default_color] * len(cells)
    
    # Constants
    MODE_TRIANGLES = 4
    VEC3 = "VEC3"
    VEC4 = "VEC4"
    
    # Initialize glTF structure
    gltf = GLTF2(asset=Asset(version="2.0"))
    gltf.materials = []
    gltf.meshes = []
    gltf.nodes = []
    gltf.accessors = []
    gltf.bufferViews = []
    
    binary_blob = bytearray()
    node_indices = []
    
    total_triangles = 0
    total_vertices = 0
    
    for cell_idx, (cell, color) in enumerate(zip(cells, colors)):
        # Get mesh data
        mesh = tf.Mesh.ByCell(cell)
        obj_content = mesh.ToOBJ()
        
        # Parse OBJ content
        vertices = []
        faces = []
        
        for line in obj_content.strip().split('\n'):
            line = line.strip()
            if line.startswith('v '):
                parts = line.split()[1:4]
                vertices.append([float(p) for p in parts])
            elif line.startswith('f '):
                parts = line.split()[1:]
                # OBJ indices are 1-based
                face_indices = [int(p.split('/')[0]) - 1 for p in parts]
                # Triangulate if needed
                for i in range(1, len(face_indices) - 1):
                    faces.append([face_indices[0], face_indices[i], face_indices[i+1]])
        
        if not vertices or not faces:
            continue
        
        # Convert to numpy arrays
        positions = np.array(vertices, dtype=np.float32)
        indices = np.array(faces, dtype=np.uint32).flatten()
        
        # Apply Y-up transformation (Z-up to Y-up)
        positions_transformed = np.column_stack([
            positions[:, 0],    # X stays X
            positions[:, 2],    # Z becomes Y
            -positions[:, 1]    # -Y becomes Z
        ]).astype(np.float32)
        
        # Create vertex colors
        vertex_colors = np.tile(np.array(color, dtype=np.float32), (len(vertices), 1))
        
        # Add position data to buffer
        pos_start = len(binary_blob)
        binary_blob.extend(positions_transformed.tobytes())
        pos_length = len(binary_blob) - pos_start
        
        # Add color data to buffer
        col_start = len(binary_blob)
        binary_blob.extend(vertex_colors.tobytes())
        col_length = len(binary_blob) - col_start
        
        # Add index data to buffer
        idx_start = len(binary_blob)
        binary_blob.extend(indices.tobytes())
        idx_length = len(binary_blob) - idx_start
        
        # Create buffer views
        pos_bv_idx = len(gltf.bufferViews)
        gltf.bufferViews.append(BufferView(
            buffer=0, byteOffset=pos_start, byteLength=pos_length, target=ARRAY_BUFFER
        ))
        
        col_bv_idx = len(gltf.bufferViews)
        gltf.bufferViews.append(BufferView(
            buffer=0, byteOffset=col_start, byteLength=col_length, target=ARRAY_BUFFER
        ))
        
        idx_bv_idx = len(gltf.bufferViews)
        gltf.bufferViews.append(BufferView(
            buffer=0, byteOffset=idx_start, byteLength=idx_length, target=ELEMENT_ARRAY_BUFFER
        ))
        
        # Create accessors
        pos_acc_idx = len(gltf.accessors)
        gltf.accessors.append(Accessor(
            bufferView=pos_bv_idx,
            byteOffset=0,
            componentType=FLOAT,
            count=len(vertices),
            type=VEC3,
            min=positions_transformed.min(axis=0).tolist(),
            max=positions_transformed.max(axis=0).tolist()
        ))
        
        col_acc_idx = len(gltf.accessors)
        gltf.accessors.append(Accessor(
            bufferView=col_bv_idx,
            byteOffset=0,
            componentType=FLOAT,
            count=len(vertices),
            type=VEC4
        ))
        
        idx_acc_idx = len(gltf.accessors)
        gltf.accessors.append(Accessor(
            bufferView=idx_bv_idx,
            byteOffset=0,
            componentType=UNSIGNED_INT,
            count=len(indices),
            type="SCALAR"
        ))
        
        # Create material
        mat_idx = len(gltf.materials)
        gltf.materials.append(Material(
            pbrMetallicRoughness={
                "baseColorFactor": list(color),
                "metallicFactor": 0.0,
                "roughnessFactor": 1.0
            },
            alphaMode="BLEND" if color[3] < 1.0 else "OPAQUE",
            doubleSided=True
        ))
        
        # Create mesh
        mesh_idx = len(gltf.meshes)
        gltf.meshes.append(Mesh(
            primitives=[Primitive(
                attributes={"POSITION": pos_acc_idx, "COLOR_0": col_acc_idx},
                indices=idx_acc_idx,
                mode=MODE_TRIANGLES,
                material=mat_idx
            )],
            name=f"Cell_{cell_idx}"
        ))
        
        # Create node
        node_idx = len(gltf.nodes)
        gltf.nodes.append(Node(mesh=mesh_idx, name=f"Cell_{cell_idx}"))
        node_indices.append(node_idx)
        
        total_vertices += len(vertices)
        total_triangles += len(faces)
    
    # Create root node with all cell nodes as children
    root_idx = len(gltf.nodes)
    gltf.nodes.append(Node(name="Root", children=node_indices))
    
    # Create scene
    gltf.scenes = [Scene(nodes=[root_idx])]
    gltf.scene = 0
    
    # Create buffer
    gltf.buffers = [Buffer(byteLength=len(binary_blob))]
    
    # Ensure output directory exists
    os.makedirs(os.path.dirname(os.path.abspath(filepath)) or ".", exist_ok=True)
    
    # Save GLB
    gltf.set_binary_blob(bytes(binary_blob))
    gltf.save_binary(filepath)
    
    if not silent:
        print(f"Exported to: {filepath}")
        print(f"  Cells: {len(cells)}")
        print(f"  Vertices: {total_vertices}")
        print(f"  Triangles: {total_triangles}")
    
    return filepath

## 5. Export to GLB

Now let's export our geometry to GLB format.

In [ ]:
if PYGLTFLIB_AVAILABLE:
    # Define colors for each cell (RGBA, 0-1 range)
    colors = [
        (0.4, 0.6, 0.9, 1.0),   # Box: blue
        (0.4, 0.9, 0.4, 1.0),   # Cylinder: green  
        (0.9, 0.9, 0.4, 1.0),   # Sphere: yellow
    ]
    
    # Export all cells to a single GLB file
    output_path = "./exported_geometry.glb"
    export_to_glb(
        cells=[box, cylinder, sphere],
        filepath=output_path,
        colors=colors
    )
    
    print(f"\nFile size: {os.path.getsize(output_path) / 1024:.1f} KB")
else:
    print("Skipping GLB export - pygltflib not available")
    print("Install with: pip install pygltflib")

## 6. Export with Per-Face Colors

This example shows how to create a more colorful export with different colors for each face.

In [ ]:
def export_cell_with_face_colors(
    cell,
    filepath,
    color_scale='viridis',
    opacity=1.0,
    silent=False
):
    """
    Export a single cell with different colors for each face.
    
    NOTE: This is a simplified implementation. A full implementation would
    use topologic_fast's native color support when available.
    """
    if not PYGLTFLIB_AVAILABLE:
        raise ImportError("pygltflib is required")
    
    import colorsys
    
    faces = cell.Faces()
    n_faces = len(faces)
    
    # Generate colors using HSV color space
    face_colors = []
    for i in range(n_faces):
        hue = i / n_faces
        r, g, b = colorsys.hsv_to_rgb(hue, 0.7, 0.9)
        face_colors.append((r, g, b, opacity))
    
    # For simplicity, export the whole cell with the first color
    # A more complete implementation would export each face separately
    export_to_glb([cell], filepath, colors=[face_colors[0]], silent=silent)
    
    if not silent:
        print(f"Exported cell with {n_faces} faces")

if PYGLTFLIB_AVAILABLE:
    # Export box with face colors
    export_cell_with_face_colors(
        box,
        "./colored_box.glb",
        opacity=0.9
    )
else:
    print("Skipping - pygltflib not available")

## 7. Export CellComplex

Export a CellComplex (multiple connected cells) to GLB.

In [ ]:
# Create a CellComplex from two adjacent boxes
box1 = tf.Cell.Box(0, 0, 0, 2, 2, 2)
box2 = tf.Cell.Box(2, 0, 0, 2, 2, 2)
box3 = tf.Cell.Box(4, 0, 0, 2, 2, 2)

cell_complex = tf.CellComplex.ByCells([box1, box2, box3])
print(f"CellComplex: {cell_complex.NumCells()} cells")

# Get individual cells from the complex
cells = cell_complex.Cells()

if PYGLTFLIB_AVAILABLE:
    # Assign different colors to each cell
    colors = [
        (1.0, 0.4, 0.4, 0.8),  # Red
        (0.4, 1.0, 0.4, 0.8),  # Green
        (0.4, 0.4, 1.0, 0.8),  # Blue
    ]
    
    export_to_glb(
        cells=cells,
        filepath="./cell_complex.glb",
        colors=colors
    )
else:
    print("Skipping - pygltflib not available")

## 8. Alternative: Export to OBJ/STL

If GLB export is not available, topologic_fast provides OBJ and STL export through the Mesh class.

In [ ]:
# Export to OBJ format (always available)
mesh = tf.Mesh.ByCell(box)

# Get OBJ content
obj_content = mesh.ToOBJ()
print("OBJ content (first 500 chars):")
print(obj_content[:500])

# Save to file
with open("./box.obj", "w") as f:
    f.write(obj_content)
print(f"\nSaved to box.obj")

In [ ]:
# Export to STL format (always available)
stl_content = mesh.ToSTL()
print("STL content (first 500 chars):")
print(stl_content[:500])

# Save to file
with open("./box.stl", "w") as f:
    f.write(stl_content)
print(f"\nSaved to box.stl")

## Summary

This notebook demonstrated:

1. **Creating Geometry** - Box, cylinder, and sphere cells using topologic_fast
2. **Mesh Generation** - Converting topological objects to triangulated meshes
3. **GLB Export** - A Python implementation for exporting to GLB format
4. **Alternative Formats** - OBJ and STL export using built-in Mesh methods

### Features Not Yet Implemented in topologic_fast:

- `tf.Cell.Torus()` - Torus cell creation
- `tf.Topology.Dictionary()` / `tf.Dictionary` - Attaching metadata to topologies
- Native GLB/glTF export
- Per-face color dictionaries

### Next Steps:

- View the exported .glb files in:
  - [glTF Viewer](https://gltf-viewer.donmccurdy.com/)
  - [Babylon.js Sandbox](https://sandbox.babylonjs.com/)
  - Windows 3D Viewer
  - Blender
  
- For AR/VR applications, consider using the exported GLB with:
  - A-Frame
  - Three.js
  - Unity/Unreal Engine